# Hurricane Melissa vs Mangrove NDVI Decline (Steps 1-4)

This notebook links FN mangrove NDVI change to Hurricane Melissa track layers.

Implemented analyses:
1. `Inside vs outside wind swath` comparison for NDVI decline
2. `Distance-to-track` analysis with distance bins
3. `Intensity/pressure-linked exposure` using nearest track points
4. `Spatial concentration` of declining mangroves by side/quadrant relative to nearest track point

Inputs:
- NDVI before/after rasters
- FN mangrove polygons
- NOAA best-track layers (`AL132025_*`)


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.ops import unary_union

plt.style.use('default')
pd.set_option('display.max_columns', 150)


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'dphil_papers').exists():
            return p
    raise FileNotFoundError(f'Could not find project root from {start}')

ROOT = find_project_root(Path.cwd())

# Core inputs
ndvi_before_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/ndvi/HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif'
ndvi_after_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/ndvi/HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif'
fn_mangroves_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp'

# Hurricane Melissa track inputs
track_dir = ROOT / 'dphil_papers/dphil_paper_3/inputs/hurricane_melissa_track_noaa/al132025_best_track'
line_path = track_dir / 'AL132025_lin.shp'
pts_path = track_dir / 'AL132025_pts.shp'
radii_path = track_dir / 'AL132025_radii.shp'
windswath_path = track_dir / 'AL132025_windswath.shp'

jamaica_boundary_path = ROOT / 'dphil_papers/dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

# Analysis settings
SUBSTANTIAL_DECLINE_THRESHOLD = -0.05   # NDVI units
DISTANCE_BINS_KM = [0, 25, 50, 75, 100, 150, 200, 300, 500, 1000]

# Export settings
output_dir = ROOT / 'dphil_papers/dphil_paper_3/results/hurricane_melissa_track_analysis'
output_dir.mkdir(parents=True, exist_ok=True)
SAVE_OUTPUTS = False

print('Project root:', ROOT)
for p in [ndvi_before_path, ndvi_after_path, fn_mangroves_path, line_path, pts_path, radii_path, windswath_path, jamaica_boundary_path]:
    print(p.name, 'exists ->', p.exists())
print('Output dir:', output_dir)


In [ ]:
def maybe_save(fig, out_path: Path, dpi=300):
    if SAVE_OUTPUTS:
        fig.savefig(out_path, dpi=dpi)
        print('Saved:', out_path)
    else:
        print('PNG export skipped (SAVE_OUTPUTS=False):', out_path.name)

def coerce_track_to_wgs84(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """NOAA files are in geographic degrees with a non-standard sphere CRS; coerce to EPSG:4326."""
    g = gdf.copy()
    b = g.total_bounds
    looks_like_lonlat = (abs(b[0]) <= 180 and abs(b[2]) <= 180 and abs(b[1]) <= 90 and abs(b[3]) <= 90)
    if looks_like_lonlat:
        g = g.set_crs(epsg=4326, allow_override=True)
        return g
    return g.to_crs(4326)

def parse_track_timestamp(df: pd.DataFrame) -> pd.Series:
    req = {'YEAR', 'MONTH', 'DAY', 'HHMM'}
    if not req.issubset(df.columns):
        return pd.Series(pd.NaT, index=df.index)

    y = pd.to_numeric(df['YEAR'], errors='coerce').astype('Int64')
    m = pd.to_numeric(df['MONTH'], errors='coerce').astype('Int64')
    d = pd.to_numeric(df['DAY'], errors='coerce').astype('Int64')
    hhmm = pd.to_numeric(df['HHMM'], errors='coerce')
    hh = (hhmm // 100).fillna(0).astype('Int64')
    mm = (hhmm % 100).fillna(0).astype('Int64')

    return pd.to_datetime(
        {'year': y, 'month': m, 'day': d, 'hour': hh, 'minute': mm},
        errors='coerce',
        utc=True,
    )

def quadrant_from_dxdy(dx, dy):
    if dx >= 0 and dy >= 0:
        return 'NE'
    if dx >= 0 and dy < 0:
        return 'SE'
    if dx < 0 and dy >= 0:
        return 'NW'
    return 'SW'


In [ ]:
# Load NDVI and FN mangroves; build paired mangrove pixel table
fn = gpd.read_file(fn_mangroves_path).to_crs(3448)
fn = fn[fn.geometry.notnull() & ~fn.geometry.is_empty].copy()

with rasterio.open(ndvi_before_path) as src_b, rasterio.open(ndvi_after_path) as src_a:
    if src_b.crs != src_a.crs or src_b.transform != src_a.transform or src_b.shape != src_a.shape:
        raise ValueError('Before/after NDVI rasters are not on the same grid.')

    arr_before = src_b.read(1)
    arr_after = src_a.read(1)
    transform = src_b.transform
    bounds = src_b.bounds

    mangrove_mask = geometry_mask(
        [g for g in fn.geometry if g is not None and not g.is_empty],
        transform=transform,
        out_shape=(src_b.height, src_b.width),
        invert=True,
    )

    valid_before = np.isfinite(arr_before) & (arr_before >= -1.0) & (arr_before <= 1.0)
    valid_after = np.isfinite(arr_after) & (arr_after >= -1.0) & (arr_after <= 1.0)

    if src_b.nodata is not None and np.isfinite(src_b.nodata):
        valid_before &= arr_before != src_b.nodata
    if src_a.nodata is not None and np.isfinite(src_a.nodata):
        valid_after &= arr_after != src_a.nodata

    valid_paired_m = valid_before & valid_after & mangrove_mask

rows, cols = np.where(valid_paired_m)
xs, ys = rasterio.transform.xy(transform, rows, cols, offset='center')

pixels = gpd.GeoDataFrame(
    {
        'row': rows,
        'col': cols,
        'ndvi_before': arr_before[valid_paired_m],
        'ndvi_after': arr_after[valid_paired_m],
    },
    geometry=gpd.points_from_xy(xs, ys),
    crs=3448,
)
pixels['ndvi_delta'] = pixels['ndvi_after'] - pixels['ndvi_before']

print('Paired mangrove pixels:', f'{len(pixels):,}')
print('Mean delta:', round(float(pixels['ndvi_delta'].mean()), 4))
print('Median delta:', round(float(pixels['ndvi_delta'].median()), 4))


In [ ]:
# Load hurricane layers and Jamaica boundary
track_line_ll = coerce_track_to_wgs84(gpd.read_file(line_path))
track_pts_ll = coerce_track_to_wgs84(gpd.read_file(pts_path))
track_radii_ll = coerce_track_to_wgs84(gpd.read_file(radii_path))
track_windswath_ll = coerce_track_to_wgs84(gpd.read_file(windswath_path))
jamaica_ll = gpd.read_file(jamaica_boundary_path).to_crs(4326)

# Project for metric operations
track_line = track_line_ll.to_crs(3448)
track_pts = track_pts_ll.to_crs(3448)
track_radii = track_radii_ll.to_crs(3448)
track_windswath = track_windswath_ll.to_crs(3448)
jamaica = jamaica_ll.to_crs(3448)

# Parse key point attributes
for c in ['INTENSITY', 'MSLP']:
    if c in track_pts.columns:
        track_pts[c] = pd.to_numeric(track_pts[c], errors='coerce')
track_pts['timestamp'] = parse_track_timestamp(track_pts)

print('Track points:', len(track_pts), 'line segments:', len(track_line), 'windswath polygons:', len(track_windswath))
print('Track point intensity range:', (track_pts['INTENSITY'].min(), track_pts['INTENSITY'].max()) if 'INTENSITY' in track_pts.columns else 'N/A')


In [ ]:
# Step 1: Inside vs outside wind swath comparison
windswath_union = unary_union(track_windswath.geometry.tolist()) if len(track_windswath) else None

if windswath_union is None or windswath_union.is_empty:
    pixels['inside_windswath'] = False
else:
    pixels['inside_windswath'] = pixels.geometry.within(windswath_union)

step1 = (
    pixels.groupby('inside_windswath')
    .agg(
        n_pixels=('ndvi_delta', 'size'),
        mean_delta=('ndvi_delta', 'mean'),
        median_delta=('ndvi_delta', 'median'),
        p25_delta=('ndvi_delta', lambda s: np.percentile(s, 25)),
        p75_delta=('ndvi_delta', lambda s: np.percentile(s, 75)),
        pct_decline_lt0=('ndvi_delta', lambda s: 100 * np.mean(s < 0)),
        pct_substantial_decline=('ndvi_delta', lambda s: 100 * np.mean(s <= SUBSTANTIAL_DECLINE_THRESHOLD)),
    )
    .reset_index()
)
step1['inside_windswath'] = step1['inside_windswath'].map({True: 'Inside wind swath', False: 'Outside wind swath'})
step1 = step1.round(4)
step1


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

for inside, color, label in [
    (True, '#d7301f', 'Inside wind swath'),
    (False, '#4575b4', 'Outside wind swath'),
]:
    vals = pixels.loc[pixels['inside_windswath'] == inside, 'ndvi_delta'].to_numpy()
    if vals.size == 0:
        continue
    bins = np.linspace(np.percentile(vals, 0.5), np.percentile(vals, 99.5), 80)
    ax.hist(vals, bins=bins, density=True, alpha=0.45, color=color, label=label)

ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.axvline(SUBSTANTIAL_DECLINE_THRESHOLD, color='black', linestyle=':', linewidth=1)
ax.set_title('Step 1: NDVI Change Inside vs Outside Hurricane Wind Swath')
ax.set_xlabel('NDVI change (after - before)')
ax.set_ylabel('Density')
ax.legend(loc='upper left')

maybe_save(fig, output_dir / 'step1_inside_outside_windswath_hist.png', dpi=300)
plt.show()


In [ ]:
# Step 2: Distance-to-track analysis (bins)
track_line_union = unary_union(track_line.geometry.tolist())
pixels['distance_to_track_km'] = pixels.geometry.distance(track_line_union) / 1000.0

bins = DISTANCE_BINS_KM + [np.inf]
labels = [f'{bins[i]}-{bins[i+1]} km' if np.isfinite(bins[i+1]) else f'{bins[i]}+ km' for i in range(len(bins)-1)]
pixels['distance_bin'] = pd.cut(pixels['distance_to_track_km'], bins=bins, labels=labels, right=False)

step2 = (
    pixels.groupby('distance_bin', observed=True)
    .agg(
        n_pixels=('ndvi_delta', 'size'),
        mean_delta=('ndvi_delta', 'mean'),
        median_delta=('ndvi_delta', 'median'),
        pct_decline_lt0=('ndvi_delta', lambda s: 100 * np.mean(s < 0)),
        pct_substantial_decline=('ndvi_delta', lambda s: 100 * np.mean(s <= SUBSTANTIAL_DECLINE_THRESHOLD)),
    )
    .reset_index()
    .round(4)
)
step2


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5), constrained_layout=True)

ax1.plot(step2['distance_bin'].astype(str), step2['median_delta'], marker='o', color='#1f78b4', label='Median delta')
ax1.axhline(0, color='black', linestyle='--', linewidth=1)
ax1.set_ylabel('Median NDVI delta')
ax1.set_xlabel('Distance bin from track')
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(step2['distance_bin'].astype(str), step2['pct_substantial_decline'], marker='s', color='#e31a1c', label='% substantial decline')
ax2.set_ylabel('% substantial decline')

ax1.set_title('Step 2: NDVI Decline vs Distance from Hurricane Track')

lines, labels = [], []
for a in [ax1, ax2]:
    l, lab = a.get_legend_handles_labels()
    lines += l
    labels += lab
ax1.legend(lines, labels, loc='upper right')

maybe_save(fig, output_dir / 'step2_distance_bin_summary_plot.png', dpi=300)
plt.show()


In [ ]:
# Step 3: Intensity/pressure-linked exposure (nearest track point)
pts_for_join = track_pts[['geometry']].copy()
for c in ['INTENSITY', 'MSLP', 'timestamp']:
    if c in track_pts.columns:
        pts_for_join[c] = track_pts[c]
pts_for_join['track_x'] = track_pts.geometry.x
pts_for_join['track_y'] = track_pts.geometry.y

pixels_exp = gpd.sjoin_nearest(
    pixels,
    pts_for_join,
    how='left',
    distance_col='distance_to_nearest_track_point_m'
)

# Intensity bins (quartiles)
if 'INTENSITY' in pixels_exp.columns and pixels_exp['INTENSITY'].notna().sum() > 0:
    pixels_exp['intensity_q'] = pd.qcut(pixels_exp['INTENSITY'], q=4, duplicates='drop')
    step3_intensity = (
        pixels_exp.groupby('intensity_q', observed=True)
        .agg(
            n_pixels=('ndvi_delta', 'size'),
            mean_delta=('ndvi_delta', 'mean'),
            median_delta=('ndvi_delta', 'median'),
            pct_substantial_decline=('ndvi_delta', lambda s: 100 * np.mean(s <= SUBSTANTIAL_DECLINE_THRESHOLD)),
        )
        .reset_index()
        .round(4)
    )
else:
    step3_intensity = pd.DataFrame()

# MSLP bins (quartiles)
if 'MSLP' in pixels_exp.columns and pixels_exp['MSLP'].notna().sum() > 0:
    pixels_exp['mslp_q'] = pd.qcut(pixels_exp['MSLP'], q=4, duplicates='drop')
    step3_mslp = (
        pixels_exp.groupby('mslp_q', observed=True)
        .agg(
            n_pixels=('ndvi_delta', 'size'),
            mean_delta=('ndvi_delta', 'mean'),
            median_delta=('ndvi_delta', 'median'),
            pct_substantial_decline=('ndvi_delta', lambda s: 100 * np.mean(s <= SUBSTANTIAL_DECLINE_THRESHOLD)),
        )
        .reset_index()
        .round(4)
    )
else:
    step3_mslp = pd.DataFrame()

print('Step 3 intensity summary:')
display(step3_intensity)
print('Step 3 MSLP summary:')
display(step3_mslp)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

if 'INTENSITY' in pixels_exp.columns and pixels_exp['INTENSITY'].notna().sum() > 0:
    hb = axes[0].hexbin(
        pixels_exp['INTENSITY'],
        pixels_exp['ndvi_delta'],
        gridsize=35,
        mincnt=1,
        cmap='viridis'
    )
    axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
    axes[0].set_xlabel('Nearest-track intensity')
    axes[0].set_ylabel('NDVI delta')
    axes[0].set_title('Step 3A: NDVI delta vs nearest intensity')
    fig.colorbar(hb, ax=axes[0], label='Pixel count')
else:
    axes[0].text(0.5, 0.5, 'No intensity field available', ha='center', va='center')
    axes[0].set_axis_off()

if 'MSLP' in pixels_exp.columns and pixels_exp['MSLP'].notna().sum() > 0:
    hb2 = axes[1].hexbin(
        pixels_exp['MSLP'],
        pixels_exp['ndvi_delta'],
        gridsize=35,
        mincnt=1,
        cmap='plasma'
    )
    axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
    axes[1].set_xlabel('Nearest-track MSLP')
    axes[1].set_ylabel('NDVI delta')
    axes[1].set_title('Step 3B: NDVI delta vs nearest pressure')
    fig.colorbar(hb2, ax=axes[1], label='Pixel count')
else:
    axes[1].text(0.5, 0.5, 'No MSLP field available', ha='center', va='center')
    axes[1].set_axis_off()

maybe_save(fig, output_dir / 'step3_intensity_pressure_relationships.png', dpi=300)
plt.show()


In [ ]:
# Step 4: Spatial concentration by side/quadrant relative to nearest track point
if 'track_x' not in pixels_exp.columns or 'track_y' not in pixels_exp.columns:
    raise ValueError('Nearest-track coordinates missing. Run Step 3 cell first.')

decline = pixels_exp[pixels_exp['ndvi_delta'] <= SUBSTANTIAL_DECLINE_THRESHOLD].copy()

decline['dx_m'] = decline.geometry.x - decline['track_x']
decline['dy_m'] = decline.geometry.y - decline['track_y']
decline['quadrant'] = [quadrant_from_dxdy(dx, dy) for dx, dy in zip(decline['dx_m'], decline['dy_m'])]
decline['side'] = np.where(decline['dx_m'] >= 0, 'East of nearest track point', 'West of nearest track point')

step4_quad = (
    decline.groupby('quadrant', observed=True)
    .agg(n_pixels=('ndvi_delta', 'size'))
    .reset_index()
)
step4_quad['pct'] = 100 * step4_quad['n_pixels'] / max(len(decline), 1)
step4_quad = step4_quad.sort_values('quadrant').reset_index(drop=True)

step4_side = (
    decline.groupby('side', observed=True)
    .agg(n_pixels=('ndvi_delta', 'size'))
    .reset_index()
)
step4_side['pct'] = 100 * step4_side['n_pixels'] / max(len(decline), 1)

print('Substantial decline pixels:', f'{len(decline):,}')
print('Quadrant concentration:')
display(step4_quad.round(3))
print('Side concentration:')
display(step4_side.round(3))


In [ ]:
# Map + bar chart for Step 4
plot_decline = decline
if len(plot_decline) > 90000:
    plot_decline = plot_decline.sample(90000, random_state=42)

quad_colors = {'NE': '#1b9e77', 'SE': '#d95f02', 'SW': '#7570b3', 'NW': '#e7298a'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
ax_map, ax_bar = axes

# Context layers
jamaica.boundary.plot(ax=ax_map, color='black', linewidth=0.8, alpha=0.8)
track_line.plot(ax=ax_map, color='#08519c', linewidth=1.8, alpha=0.9)
if len(track_windswath) > 0:
    track_windswath.plot(ax=ax_map, color='#9ecae1', alpha=0.18, edgecolor='none')

for q, col in quad_colors.items():
    sub = plot_decline[plot_decline['quadrant'] == q]
    if len(sub) > 0:
        sub.plot(ax=ax_map, markersize=3, color=col, alpha=0.55, label=q)

ax_map.set_title('Step 4: Substantial NDVI Declines by Quadrant')
ax_map.set_xlabel('Easting (m, EPSG:3448)')
ax_map.set_ylabel('Northing (m, EPSG:3448)')
ax_map.set_aspect('equal')
ax_map.legend(title='Quadrant', loc='upper right', frameon=True, framealpha=0.95)

ax_bar.bar(step4_quad['quadrant'], step4_quad['pct'], color=[quad_colors.get(q, '#999999') for q in step4_quad['quadrant']])
ax_bar.set_title('Concentration of substantial declines by quadrant')
ax_bar.set_xlabel('Quadrant')
ax_bar.set_ylabel('% of substantial decline pixels')

for i, v in enumerate(step4_quad['pct']):
    ax_bar.text(i, v + 0.5, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

maybe_save(fig, output_dir / 'step4_quadrant_concentration_map_and_bar.png', dpi=300)
plt.show()


In [ ]:
# Optional exports (tables)
if SAVE_OUTPUTS:
    step1.to_csv(output_dir / 'step1_inside_outside_windswath.csv', index=False)
    step2.to_csv(output_dir / 'step2_distance_bins.csv', index=False)
    if len(step3_intensity) > 0:
        step3_intensity.to_csv(output_dir / 'step3_intensity_bins.csv', index=False)
    if len(step3_mslp) > 0:
        step3_mslp.to_csv(output_dir / 'step3_mslp_bins.csv', index=False)
    step4_quad.to_csv(output_dir / 'step4_quadrant_concentration.csv', index=False)
    step4_side.to_csv(output_dir / 'step4_side_concentration.csv', index=False)
    print('Saved CSV summaries to:', output_dir)
else:
    print('CSV export skipped (SAVE_OUTPUTS=False)')


## Notes
- `SUBSTANTIAL_DECLINE_THRESHOLD = -0.05` is configurable.
- Quadrants are defined relative to each pixel's nearest track point using projected dx/dy.
- NOAA layer CRS is coerced to EPSG:4326, then projected to EPSG:3448 for distance/overlay analyses.
